# 04 - Ranking Model

Train an XGBoost classifier to predict hire/reject decisions based on skill match, experience fit, and semantic similarity.

In [ ]:
!pip install -q pandas numpy scikit-learn xgboost joblib matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries loaded")

In [ ]:
# Load datasets
candidates = pd.read_csv('../datasets/candidates.csv')
jobs = pd.read_csv('../datasets/jobs.csv')

print(f"Candidates: {len(candidates)}, Jobs: {len(jobs)}")

In [ ]:
# Generate training data: create candidate-job pairs with labels
np.random.seed(42)

def create_training_data(candidates, jobs, n_samples=2000):
    data = []
    
    for _ in range(n_samples):
        cand = candidates.sample(1).iloc[0]
        job = jobs.sample(1).iloc[0]
        
        # Parse skills
        cand_skills = set(s.strip().lower() for s in cand['skills'].split(','))
        job_skills = set(s.strip().lower() for s in job['required_skills'].split(','))
        
        # Calculate features
        skill_overlap = len(cand_skills & job_skills) / len(job_skills) if job_skills else 0
        exp_diff = cand['experience'] - job['min_experience']
        exp_match = min(1.0, max(0.0, 1 + exp_diff / 5))  # Normalize
        
        # Simple keyword-based semantic score
        cand_text = cand['resume_text'].lower()
        semantic = sum(1 for s in job_skills if s in cand_text) / len(job_skills) if job_skills else 0
        
        # Generate label based on realistic criteria
        hire_score = skill_overlap * 0.4 + exp_match * 0.3 + semantic * 0.3
        label = 1 if hire_score > 0.5 else 0
        # Add some noise for realism
        if np.random.random() < 0.1:
            label = 1 - label
        
        data.append({
            'skill_score': skill_overlap * 100,
            'exp_score': exp_match * 100,
            'semantic_score': semantic * 100,
            'experience_years': cand['experience'],
            'min_experience': job['min_experience'],
            'label': label
        })
    
    return pd.DataFrame(data)

train_df = create_training_data(candidates, jobs, n_samples=2000)
print(f"Training data: {len(train_df)} samples")
print(f"Label distribution: {train_df['label'].value_counts().to_dict()}")

In [ ]:
# Prepare features and labels
feature_cols = ['skill_score', 'exp_score', 'semantic_score', 'experience_years', 'min_experience']
X = train_df[feature_cols].values
y = train_df['label'].values

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# Train XGBoost model
model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)

model.fit(X_train_scaled, y_train)
print("Model trained successfully")

In [ ]:
# Evaluate model
y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.3f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Reject', 'Hire']))

In [ ]:
# Feature importance
importances = model.feature_importances_
feat_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': importances
}).sort_values('importance', ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(data=feat_importance, x='importance', y='feature', palette='Blues_r')
plt.title('Feature Importance (XGBoost)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

print("Feature importance:")
for _, row in feat_importance.iterrows():
    print(f"  {row['feature']}: {row['importance']:.3f}")

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Reject', 'Hire'], yticklabels=['Reject', 'Hire'])
plt.title('Confusion Matrix')
plt.ylabel('True')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

In [ ]:
# Save model and scaler
os.makedirs('../backend/app/ml_models', exist_ok=True)

joblib.dump(model, '../backend/app/ml_models/ranking_model.pkl')
joblib.dump(scaler, '../backend/app/ml_models/scaler.pkl')

print("Saved ranking_model.pkl and scaler.pkl")
print("Training complete!")